# Kelly vs. SBF vs. VN vs. risk-neutral — a far-OTM call option

A realistic far-out-of-the-money index call: each round the player buys a one-month OTM call on a broad index. With probability $p = 0.10$ the option finishes in the money and pays $13\times$ the premium; with probability $0.90$ it expires worthless. Per dollar of premium, the gross return is
$$R = \begin{cases} 13 & \text{w.p. } 0.10 \\ 0 & \text{w.p. } 0.90 \end{cases}$$
The arithmetic per-dollar EV is $0.1 \cdot 13 - 1 = +0.30$ (a 30% per-round expected return — favorable but not absurdly so, broadly consistent with vol-risk-premium estimates on far-OTM tail hedges in some sub-periods). The lose state is full wipeout, and it hits 90% of the time. Each player bets a fraction $f$ of wealth each round, so wealth evolves as
$$W_{t+1} \;=\; W_t \bigl(1 + f\,(R - 1)\bigr).$$
We solve for the optimal $f$ for **four CRRA agents** at $\gamma \in \{1, 0.75, 0.5, 0.25\}$ and simulate $N = 10{,}000$ players for each:

- **Kelly** ($\gamma = 1$, log utility) — the geometric-growth optimum.
- **SBF** ($\gamma = 0.75$, $K = 16$ on a 50-50 ruin gamble) — Sam Bankman-Fried's stated risk preference.
- **$\gamma = 0.5$** ($K = 4$) — the breakeven: at this $\gamma$ the agent's optimal fraction yields *zero* expected log growth on $\{R, 0\}$ binary lotteries (in the small-$f$ limit, this is exact: $g(\gamma) \propto (2\gamma - 1)/\gamma^2$).
- **VN** ($\gamma = 0.25$, $K \approx 2.5$) — Victor Niederhoffer, named for the legendary commodities trader who blew up at least twice (1997, 2007) writing deep-OTM puts. This represents an agent with so little risk aversion she'd take a 50-50 bet at just $2.5\times$ upside.
- **Risk neutral** ($f = 1$) — maximizes arithmetic $E[W_T]$ by going all-in each round; one losing round wipes you out.

In [1]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize_scalar

rng = np.random.default_rng(544)

In [2]:
initial_investment = 1000
T = 100         # rounds per player
N = 10_000      # players per strategy

# Lottery: P(R = 13) = 0.10, P(R = 0) = 0.90.  Per-dollar EV = +0.30.
returns = np.array([13.0, 0.0])
probs   = np.array([0.10, 0.90])
p, x_win, x_lose = 0.10, 13.0, 0.0

In [3]:
def crra_utility(w, gamma):
    if gamma == 1:
        return np.log(w) if w > 0 else -np.inf
    if w <= 0:
        return -1.0 / (1 - gamma) if 1 - gamma > 0 else -np.inf
    return (w ** (1 - gamma) - 1) / (1 - gamma)

def find_f_star(p, x1, x2, gamma):
    def neg_eu(f):
        return -(p * crra_utility(1 + f * (x1 - 1), gamma)
                 + (1 - p) * crra_utility(1 + f * (x2 - 1), gamma))
    return minimize_scalar(neg_eu, bounds=(1e-9, 1 - 1e-9), method="bounded").x

def expected_log_growth(f, p, x1, x2):
    if f >= 1.0 - 1e-9 and x2 == 0.0:
        return -np.inf
    return p * np.log(1 + f * (x1 - 1)) + (1 - p) * np.log(1 + f * (x2 - 1))

agents = [
    ("Kelly (gamma=1)",  1.00),
    ("SBF (gamma=0.75)", 0.75),
    ("gamma=0.5",        0.50),
    ("VN (gamma=0.25)",  0.25),
]

f_table_rows = []
for label, gamma in agents:
    f = find_f_star(p, x_win, x_lose, gamma)
    g = expected_log_growth(f, p, x_win, x_lose)
    f_table_rows.append({"agent": label, "gamma": gamma, "f*": f, "E[log growth]": g})
# Risk-neutral: f = 1, expected log growth is -inf (full wipeout w.p. 0.9 per round)
f_table_rows.append({"agent": "Risk neutral", "gamma": 0.0, "f*": 1.0,
                     "E[log growth]": -np.inf})

f_table = pd.DataFrame(f_table_rows)
print(f_table.to_string(index=False, float_format="{:.4f}".format))

# Pull out the fractions we'll use for simulation
f_kelly = f_table_rows[0]["f*"]
f_sbf   = f_table_rows[1]["f*"]
f_half  = f_table_rows[2]["f*"]
f_vn    = f_table_rows[3]["f*"]
f_rn    = 1.0

           agent  gamma     f*  E[log growth]
 Kelly (gamma=1) 1.0000 0.0250         0.0035
SBF (gamma=0.75) 0.7500 0.0347         0.0030
       gamma=0.5 0.5000 0.0565        -0.0006
 VN (gamma=0.25) 0.2500 0.1425        -0.0387
    Risk neutral 0.0000 1.0000           -inf


## Solve for each agent's optimal fraction

Each CRRA agent picks $f$ to maximize $E[u_\gamma(W_0 (1 + f(R-1)))]$ where
$$u_\gamma(w) = \begin{cases} \log w & \gamma = 1 \\ \dfrac{w^{1-\gamma} - 1}{1 - \gamma} & \gamma \neq 1 \end{cases}$$
For $\gamma < 1$, $u_\gamma(0)$ is finite ($-1/(1-\gamma)$), so the agent does *not* infinitely penalize ruin — though her $f^*$ stays interior because partial exposure dominates the all-in corner.

In [4]:
# Vectorized Monte Carlo: each row is one player's T draws of the underlying asset.
draws = rng.choice(returns, size=(N, T), p=probs)

def simulate(f):
    return initial_investment * np.prod(1 + f * (draws - 1), axis=1)

kelly_final = simulate(f_kelly)
sbf_final   = simulate(f_sbf)
half_final  = simulate(f_half)
vn_final    = simulate(f_vn)
rn_final    = simulate(f_rn)

## Expected final wealth, survival, and "above start" rates

Returns are i.i.d., so $E[W_T] = W_0 \cdot (E[\text{multiplier}])^T$ with $E[1 + f(R-1)] = 1 + f \cdot 0.30$. Three diagnostics tell the story:

- **Theoretical $E[W_T]$** vs **simulated $E[W_T]$** — these diverge dramatically when ruin is the modal outcome.
- **Survival rate** — fraction of players with $W_T > 0$.
- **Above-start rate** — fraction of players with $W_T > W_0$.

In [5]:
E_R = float(np.dot(probs, returns))   # arithmetic mean of R

def theoretical_E_WT(f):
    return initial_investment * (1 + f * (E_R - 1)) ** T

def fmt_dollars(x):
    return f"${x:,.0f}"

def diagnostics(final, f, label):
    return {
        "Strategy":            label,
        "Theoretical E[W_T]":  fmt_dollars(theoretical_E_WT(f)),
        "Simulated E[W_T]":    fmt_dollars(final.mean()),
        "Survival rate":       f"{(final > 0).mean():.0%}",
        "Above-start rate":    f"{(final > initial_investment).mean():.0%}",
    }

ev_table = pd.DataFrame([
    diagnostics(kelly_final, f_kelly, f"Kelly  (f={f_kelly:.4f})"),
    diagnostics(sbf_final,   f_sbf,   f"SBF    (f={f_sbf:.4f})"),
    diagnostics(half_final,  f_half,  f"g=0.5  (f={f_half:.4f})"),
    diagnostics(vn_final,    f_vn,    f"VN     (f={f_vn:.4f})"),
    diagnostics(rn_final,    f_rn,    f"Risk-neutral (f=1)"),
])
print(ev_table.to_string(index=False))

          Strategy   Theoretical E[W_T] Simulated E[W_T] Survival rate Above-start rate
 Kelly  (f=0.0250)               $2,111           $2,125          100%              68%
 SBF    (f=0.0347)               $2,818           $2,833          100%              56%
 g=0.5  (f=0.0565)               $5,362           $5,222          100%              42%
 VN     (f=0.1425)              $65,784          $15,392          100%              13%
Risk-neutral (f=1) $247,933,511,096,598               $0            0%               0%


## Percentiles of the wealth distribution

In [6]:
percentiles = [1, 5, 10, 25, 50, 75, 90, 95, 99, 99.9, 99.99]

def fmt(x):
    return f"${x:,.0f}"

table = pd.DataFrame({
    "Percentile":   percentiles,
    "Kelly":        [fmt(v) for v in np.percentile(kelly_final, percentiles)],
    "SBF":          [fmt(v) for v in np.percentile(sbf_final,   percentiles)],
    "g=0.5":        [fmt(v) for v in np.percentile(half_final,  percentiles)],
    "VN":           [fmt(v) for v in np.percentile(vn_final,    percentiles)],
    "Risk neutral": [fmt(v) for v in np.percentile(rn_final,    percentiles)],
})
print(table.to_string(index=False))

 Percentile   Kelly      SBF    g=0.5          VN Risk neutral
       1.00    $251     $135      $30          $0           $0
       5.00    $335     $199      $53          $0           $0
      10.00    $447     $292      $95          $0           $0
      25.00    $794     $628     $299          $2           $0
      50.00  $1,412   $1,353     $944         $21           $0
      75.00  $2,510   $2,915   $2,985        $209           $0
      90.00  $4,463   $6,277   $9,434      $2,087           $0
      95.00  $5,950   $9,212  $16,772      $6,597           $0
      99.00 $14,105  $29,113  $94,240    $208,252           $0
      99.90 $25,075  $62,700 $297,852  $2,080,172           $0
      99.99 $44,577 $135,032 $941,379 $20,778,270           $0


## Top 10 final wealth values among the 10,000 players (each strategy)

In [7]:
def top10(final):
    return np.sort(final)[::-1][:10]

top10_table = pd.DataFrame({
    "Rank":         np.arange(1, 11),
    "Kelly":        [fmt(v) for v in top10(kelly_final)],
    "SBF":          [fmt(v) for v in top10(sbf_final)],
    "g=0.5":        [fmt(v) for v in top10(half_final)],
    "VN":           [fmt(v) for v in top10(vn_final)],
    "Risk neutral": [fmt(v) for v in top10(rn_final)],
})
print(top10_table.to_string(index=False))

 Rank   Kelly      SBF    g=0.5          VN Risk neutral
    1 $44,577 $135,032 $941,379 $20,778,270           $0
    2 $44,577 $135,032 $941,379 $20,778,270           $0
    3 $33,433  $92,013 $529,520  $6,574,373           $0
    4 $33,433  $92,013 $529,520  $6,574,373           $0
    5 $33,433  $92,013 $529,520  $6,574,373           $0
    6 $33,433  $92,013 $529,520  $6,574,373           $0
    7 $33,433  $92,013 $529,520  $6,574,373           $0
    8 $33,433  $92,013 $529,520  $6,574,373           $0
    9 $33,433  $92,013 $529,520  $6,574,373           $0
   10 $25,075  $62,700 $297,852  $2,080,172           $0


**Takeaway.** Each agent picks $f$ optimally for *her* risk aversion. The lottery is positive-EV (per-dollar +30%), so a risk-neutral agent goes all-in and is wiped out with certainty. Less risk-averse-than-log agents over-bet relative to Kelly and pay for it in compounded performance:

- **Kelly** ($\gamma = 1$, $f^* \approx 0.025$): expected log growth $+0.0035$. Median wealth grows ~40%; 68% of players end above start.
- **SBF** ($\gamma = 0.75$, $f^* \approx 0.035$): expected log growth $+0.0030$. SBF's $f$ is ~40% larger than Kelly's; her median is similar but her left tail is meaningfully worse (1st-percentile player at \$135 vs Kelly's \$251).
- **$\gamma = 0.5$** ($f^* \approx 0.057$): expected log growth $\approx 0$ — by design, this is the threshold of the CRRA family where the optimal $f$ produces zero compounded growth. Median ends almost exactly at starting wealth (\$944). Above-start rate falls to 42%.
- **VN** ($\gamma = 0.25$, $f^* \approx 0.143$): expected log growth $-0.039$. Median wealth shrinks by a factor $e^{-3.87} \approx 0.021$ — the typical Niederhoffer-style player, betting "optimally" for her preferences, ends with about 2% of her starting wealth. The right tail can still be huge (top of 10,000: \$21M) but only 13% of VN players end above start.
- **Risk-neutral** ($f = 1$): theoretical $E[W_T] \approx 10^{14}$, simulated $E[W_T] = 0$. With probability $1 - 0.1^{100} \approx 1$, every player is wiped out.

The lesson is clean and structural: in the CRRA family on a binary $\{R, 0\}$ lottery, the agent's expected log growth at her own optimum is approximately $g_K \cdot (2\gamma - 1)/\gamma^2$. That factor is positive for $\gamma > 0.5$ (Kelly, SBF), zero at $\gamma = 0.5$, and negative for $\gamma < 0.5$ (VN, risk-neutral). $\gamma = 0.5$ is the dividing line: optimization with less risk aversion than that produces *expected* wealth shrinkage even when the underlying gamble has strongly positive EV.